# CLMS Data Downloader Example
This notebook demonstrates how to access and download data from the Copernicus Land Monitoring Service (CLMS), a key component of the Copernicus Earth observation program. CLMS provides consistent and high-quality information on land cover, land use, and land dynamics across Europe, supporting environmental and policy monitoring.


To download the data, APIs can be used. However, we found that the easyest way to download the data is to use the `Provider's manifest`, which is a list of links that when opened, the available files are downloaded. Each link consists of a file. For example, for the FAPAR variable (https://land.copernicus.eu/en/products/vegetation/fraction-of-absorbed-photosynthetically-active-radiation-v2-0-1km#download), the manifest can be opened [here](https://globalland.vito.be/download/manifest/fapar_1km_v2_10daily_netcdf/manifest_clms_global_fapar_1km_v2_10daily_netcdf_latest.txt) and it contains a list like this one:

```

https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20170610/c_gls_FAPAR-RT0_201706100000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20170620/c_gls_FAPAR-RT0_201706200000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20170630/c_gls_FAPAR-RT0_201706300000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20170710/c_gls_FAPAR-RT0_201707100000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20170720/c_gls_FAPAR-RT0_201707200000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20170731/c_gls_FAPAR-RT0_201707310000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20170810/c_gls_FAPAR-RT0_201708100000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20170820/c_gls_FAPAR-RT0_201708200000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20170831/c_gls_FAPAR-RT0_201708310000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20170910/c_gls_FAPAR-RT0_201709100000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20170920/c_gls_FAPAR-RT0_201709200000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20170930/c_gls_FAPAR-RT0_201709300000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20171010/c_gls_FAPAR-RT0_201710100000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20171020/c_gls_FAPAR-RT0_201710200000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20171031/c_gls_FAPAR-RT0_201710310000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20171110/c_gls_FAPAR-RT0_201711100000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20171120/c_gls_FAPAR-RT0_201711200000_GLOBE_PROBAV_V2.0.1.nc
https://globalland.vito.be/download/netcdf/fraction_absorbed_par/fapar_1km_v2_10daily/2017/20171130/c_gls_FAPAR-RT0_201711300000_GLOBE_PROBAV_V2.0.1.nc

...

```

After saving the manifest on a `.txt` file, for each link on the manifest we download the complete file. Those downloaded raw files consist of global data, therefore, they need to be cropped to Spain and then stored. Therefore, there are 3 steps:

1. Download the raw files
2. Clip the data to Spain
3. Delete the original file to save disk space

### 1. Download raw files

In [ ]:
import requests
import os

var_name = "fapar"

folder_path = "...../FAPAR_folder" 
link_path = folder_path + "/links.txt"

with open(link_path, "r") as file:
    urls = file.readlines()

for url in urls:
    url = url.strip()
    text = url.split("/")[-2]
    year, month, day = text[:4], text[4:6], text[6:]

    global_filename = f"global_{var_name}_{year}_{month}_{day}.nc"
    global_file_path = folder_path + "/" + global_filename

    clipped_file = folder_path + f"/{var_name}_peninsula_{year}_{month}_{day}.nc"
    if os.path.exists(clipped_file):
        print(f"Files already processed:\n\t{clipped_file}")
        continue
    else:
        response = requests.get(url)
        with open(global_file_path, 'wb') as f:
            f.write(response.content)    
        print(f"Dataset downloaded: {global_file_path}")
        

### 2. Clip the data to Spain

In [ ]:
import xarray as xr

for url in urls:
    url = url.strip()
    text = url.split("/")[-2]
    year, month, day = text[:4], text[4:6], text[6:]

    global_filename = f"global_{var_name}_{year}_{month}_{day}.nc"
    global_file_path = folder_path + "/" + global_filename

    clipped_file = folder_path + f"/{var_name}_peninsula_{year}_{month}_{day}.nc"
    
    if os.path.exists(clipped_file):
        print(f"Files already processed:\n\t{clipped_file}")
        continue
    else:
        with xr.open_dataset(global_file_path) as ds:
            # Clip it to spain:
            ds_clipped_spain = ds.sel(lat=slice(44, 35.7), lon=slice(-10, 5)) # Clip region
            ds_clipped_spain = ds_clipped_spain.drop_vars(["LENGTH_AFTER", "LENGTH_BEFORE",
                                            "NOBS", "RMSE", "QFLAG"]) # Delete non useful variables

            ds_clipped_spain.to_netcdf(clipped_file)

            print(f"Clipped:\n{clipped_file}")

### 3. Delete the raw file

In [ ]:
for url in urls:
    url = url.strip()
    text = url.split("/")[-2]
    year, month, day = text[:4], text[4:6], text[6:]

    global_filename = f"global_{var_name}_{year}_{month}_{day}.nc"
    global_file_path = folder_path + "/" + global_filename

    clipped_file_peninsula = folder_path + f"/{var_name}_peninsula_{year}_{month}_{day}.nc"
    if os.path.exists(clipped_file_peninsula) and os.path.exists(global_file_path):
        os.remove(global_file_path) 
        print(f"{global_file_path} has been deleted.")
        